## Incorporações

No nosso exemplo anterior, operámos sobre vetores de sacos de palavras de alta dimensionalidade com comprimento `vocab_size`, e estávamos a converter explicitamente de vetores de representação posicional de baixa dimensionalidade para uma representação esparsa one-hot. Esta representação one-hot não é eficiente em termos de memória, para além disso, cada palavra é tratada de forma independente umas das outras, ou seja, vetores codificados one-hot não exprimem nenhuma semelhança semântica entre palavras.

Nesta unidade, iremos continuar a explorar o conjunto de dados **News AG**. Para começar, vamos carregar os dados e obter algumas definições do notebook anterior.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## O que é embedding?

A ideia de **embedding** é representar palavras por vetores densos de baixa dimensão, que de alguma forma refletem o significado semântico de uma palavra. Mais tarde, iremos discutir como construir embeddings de palavras significativos, mas para já vamos pensar em embeddings como uma forma de reduzir a dimensionalidade de um vetor de palavras.

Portanto, uma camada de embedding receberia uma palavra como entrada e produziria um vetor de saída com a dimensão especificada `embedding_size`. Por assim dizer, é muito semelhante a uma camada `Linear`, mas em vez de receber um vetor codificado one-hot, será capaz de aceitar um número da palavra como entrada.

Ao usar a camada de embedding como a primeira camada na nossa rede, podemos passar do modelo bag-of-words para um modelo de **embedding bag**, onde primeiro convertemos cada palavra no nosso texto para o embedding correspondente, e depois calculamos alguma função agregada sobre todos esses embeddings, como `sum`, `average` ou `max`.  

![Imagem a mostrar um classificador de embedding para cinco palavras numa sequência.](../../../../../translated_images/pt-PT/embedding-classifier-example.b77f021a7ee67eee.webp)

A nossa rede neural classificadora começará com uma camada de embedding, seguida por uma camada de agregação, e um classificador linear por cima:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### Lidando com tamanho variável da sequência

Como resultado desta arquitetura, os minibatches para a nossa rede precisariam de ser criados de uma certa forma. Na unidade anterior, ao usar bag-of-words, todos os tensores BoW num minibatch tinham tamanho igual `vocab_size`, independentemente do comprimento real da nossa sequência de texto. Uma vez que passamos para word embeddings, acabaríamos com um número variável de palavras em cada amostra de texto, e ao combinar essas amostras em minibatches teríamos de aplicar algum preenchimento.

Isto pode ser feito usando a mesma técnica de fornecer a função `collate_fn` à fonte de dados:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### Treinamento do classificador de embeddings

Agora que definimos o dataloader adequado, podemos treinar o modelo usando a função de treino que definimos na unidade anterior:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **Nota**: Estamos apenas a treinar para 25k registos aqui (menos de uma época completa) por questão de tempo, mas pode continuar a treinar, escrever uma função para treinar durante várias épocas, e experimentar com o parâmetro da taxa de aprendizagem para alcançar uma maior precisão. Deve ser capaz de chegar a uma precisão de cerca de 90%.


### Camada EmbeddingBag e Representação de Sequências de Comprimento Variável

Na arquitetura anterior, precisávamos de preencher todas as sequências até ao mesmo comprimento para as encaixar num minibatch. Esta não é a forma mais eficiente de representar sequências de comprimento variável - outra abordagem seria usar um vetor de **offset**, que guardaria os offsets de todas as sequências armazenadas num vetor grande.

![Imagem a mostrar uma representação de sequência por offset](../../../../../translated_images/pt-PT/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **Nota**: Na imagem acima, mostramos uma sequência de caracteres, mas no nosso exemplo estamos a trabalhar com sequências de palavras. No entanto, o princípio geral de representar sequências com um vetor offset mantém-se o mesmo.

Para trabalhar com representação por offset, usamos a camada [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html). É semelhante a `Embedding`, mas recebe um vetor de conteúdo e um vetor de offset como entrada, e inclui também uma camada de média, que pode ser `mean`, `sum` ou `max`.

Aqui está a rede modificada que usa `EmbeddingBag`:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

Para preparar o conjunto de dados para treino, precisamos fornecer uma função de conversão que irá preparar o vetor de deslocamento:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

Note que, ao contrário de todos os exemplos anteriores, a nossa rede agora aceita dois parâmetros: vetor de dados e vetor de deslocamento, que têm tamanhos diferentes. De forma semelhante, o nosso carregador de dados também nos fornece 3 valores em vez de 2: tanto os vetores de texto como de deslocamento são fornecidos como características. Portanto, precisamos ajustar ligeiramente a nossa função de treino para cuidar disso:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## Embeddings Semânticos: Word2Vec

No nosso exemplo anterior, a camada de embedding do modelo aprendeu a mapear palavras para uma representação vetorial, no entanto, esta representação não tinha muito significado semântico. Seria interessante aprender uma representação vetorial em que palavras semelhantes ou sinónimos corresponderiam a vetores próximos uns dos outros em termos de alguma distância vetorial (ex. distância euclidiana).

Para isso, precisamos pré-treinar o nosso modelo de embedding numa grande coleção de texto de uma forma específica. Uma das primeiras formas de treinar embeddings semânticos chama-se [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). Baseia-se em duas arquiteturas principais que são usadas para produzir uma representação distribuída das palavras:

 - **Continuous bag-of-words** (CBoW) — nesta arquitetura, treinamos o modelo para prever uma palavra a partir do contexto envolvente. Dado o ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$, o objetivo do modelo é prever $W_0$ a partir de $(W_{-2},W_{-1},W_1,W_2)$.
 - **Continuous skip-gram** é o oposto do CBoW. O modelo usa a janela de contexto envolvente para prever a palavra atual.

O CBoW é mais rápido, enquanto o skip-gram é mais lento, mas faz um trabalho melhor na representação de palavras pouco frequentes.

![Imagem a mostrar os algoritmos CBoW e Skip-Gram para converter palavras em vetores.](../../../../../translated_images/pt-PT/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Para experimentar com o embedding word2vec pré-treinado no conjunto de dados Google News, podemos usar a biblioteca **gensim**. Abaixo encontramos as palavras mais semelhantes a 'neural'

> **Nota:** Quando cria os vetores de palavras pela primeira vez, o download pode demorar algum tempo!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


Também podemos calcular embeddings vetoriais a partir da palavra, para serem usados no treino de um modelo de classificação (mostramos apenas os primeiros 20 componentes do vetor para maior clareza):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

A grande vantagem das embeddings semânticas é que pode manipular a codificação vetorial para alterar a semântica. Por exemplo, podemos pedir para encontrar uma palavra cuja representação vetorial seja o mais próxima possível das palavras *king* e *woman*, e o mais distante possível da palavra *man*:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

Tanto o CBoW como o Skip-Grams são embeddings “previsivos”, no sentido em que apenas consideram contextos locais. O Word2Vec não aproveita o contexto global. 

**FastText**, baseia-se no Word2Vec ao aprender representações vetoriais para cada palavra e para os n-gramas de caracteres encontrados em cada palavra. Os valores das representações são depois promediados num único vetor em cada passo de treino. Embora isto acrescente muita computação adicional ao pré-treino, permite que os embeddings de palavras codifiquem informação sub-palavra. 

Outro método, o **GloVe**, aproveita a ideia da matriz de co-ocorrência e usa métodos neurais para decompor a matriz de co-ocorrência em vetores de palavras mais expressivos e não lineares.

Pode experimentar o exemplo mudando os embeddings para FastText e GloVe, uma vez que o gensim suporta vários modelos diferentes de embeddings de palavras.


## Utilizar Embeddings Pré-Treinados em PyTorch

Podemos modificar o exemplo acima para pré-preencher a matriz na nossa camada de embeddings com embeddings semânticos, tais como Word2Vec. Precisamos ter em conta que os vocabulários do embedding pré-treinado e do nosso corpus de texto provavelmente não coincidirão, pelo que iremos inicializar os pesos para as palavras em falta com valores aleatórios:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


Agora vamos treinar o nosso modelo. Note que o tempo que leva para treinar o modelo é significativamente maior do que no exemplo anterior, devido ao tamanho maior da camada de embedding e, portanto, um número muito maior de parâmetros. Além disso, por causa disso, pode ser necessário treinar o nosso modelo com mais exemplos se quisermos evitar overfitting.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

No nosso caso, não vemos um grande aumento na precisão, o que provavelmente se deve a vocabulários bastante diferentes. 
Para superar o problema dos vocabulários diferentes, podemos usar uma das seguintes soluções:
* Re-treinar o modelo word2vec no nosso vocabulário
* Carregar o nosso conjunto de dados com o vocabulário do modelo word2vec pré-treinado. O vocabulário usado para carregar o conjunto de dados pode ser especificado durante o carregamento.

A última abordagem parece mais fácil, especialmente porque o framework PyTorch `torchtext` contém suporte incorporado para embeddings. Podemos, por exemplo, instanciar um vocabulário baseado em GloVe da seguinte maneira:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


O vocabulário carregado tem as seguintes operações básicas:
* `vocab.stoi` dicionário permite-nos converter uma palavra no seu índice de dicionário
* `vocab.itos` faz o oposto - converte um número em palavra
* `vocab.vectors` é o array de vetores embedding, por isso para obter o embedding de uma palavra `s` precisamos usar `vocab.vectors[vocab.stoi[s]]`

Aqui está o exemplo de manipulação de embeddings para demonstrar a equação **kind-man+woman = queen** (tive de ajustar um pouco o coeficiente para que funcionasse):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

Para treinar o classificador usando esses embeddings, primeiro precisamos codificar o nosso conjunto de dados usando o vocabulário GloVe:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

Como vimos acima, todos os embeddings de vetores estão armazenados na matriz `vocab.vectors`. Isso torna muito fácil carregar esses pesos para os pesos da camada de embedding usando uma simples cópia:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

Agora vamos treinar o nosso modelo e ver se obtemos melhores resultados:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

Uma das razões pelas quais não estamos a ver um aumento significativo na precisão deve-se ao facto de algumas palavras do nosso conjunto de dados estarem ausentes no vocabulário pré-treinado do GloVe, e por isso são essencialmente ignoradas. Para ultrapassar este facto, podemos treinar as nossas próprias embeddings no nosso conjunto de dados. 


## Embeddings Contextuais

Uma limitação fundamental das representações tradicionais pré-treinadas de embeddings como o Word2Vec é o problema da desambiguação do sentido das palavras. Embora os embeddings pré-treinados possam captar parte do significado das palavras no contexto, cada possível significado de uma palavra é codificado no mesmo embedding. Isto pode causar problemas nos modelos subsequentes, uma vez que muitas palavras, como a palavra 'play', têm significados diferentes dependendo do contexto em que são usadas.

Por exemplo, a palavra 'play' nestas duas frases diferentes tem um significado bastante diferente:
- Fui a uma **peça** de teatro.
- O John quer **brincar** com os amigos.

Os embeddings pré-treinados acima representam ambos os significados da palavra 'play' no mesmo embedding. Para ultrapassar esta limitação, precisamos de construir embeddings baseados no **modelo de linguagem**, que é treinado num grande corpus de texto, e que *sabe* como as palavras podem ser combinadas em diferentes contextos. Discutir embeddings contextuais está fora do âmbito deste tutorial, mas voltaremos a eles quando falarmos de modelos de linguagem na próxima unidade.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
